# Transaction Foundation Model — pretraining to serving on Ray

<div align="left">
  <a target="_blank" href="https://console.anyscale.com/template-preview/fintech_transaction_fm"><img src="https://img.shields.io/badge/🚀 Run_on-Anyscale-9hf"></a>&nbsp;
  <a href="https://github.com/anyscale/templates/tree/main/templates/fintech_transaction_fm" role="button"><img src="https://img.shields.io/static/v1?label=&message=View%20On%20GitHub&color=586069&logo=github&labelColor=2f363d"></a>&nbsp;
</div>

**⏱️ Time to complete**: 45 min

### Anyscale Technical Demo — Ray Data + Ray Train + Ray Serve

---

## Business Context

Banks and fintechs are converging on **transaction foundation models** (TFMs): a single self-supervised transformer pretrained on raw transaction sequences, producing a reusable **customer embedding** that powers fraud, churn, credit, and personalization — replacing dozens of hand-built feature pipelines. Stripe, Visa (TREASURE), Nubank, and Revolut (PRAGMA) have all published variants of this recipe.

The model itself is small and not the hard part. The hard parts are **engineering at scale**: tokenizing petabytes of transactions, pretraining across many GPUs, and re-embedding every customer on a schedule — then serving those embeddings both in batch and in real time.

**This template** builds the whole pipeline on Ray, with one upgrade over the standard NVIDIA blueprint: a **static/dynamic field split** in the tokenizer and model (the idea behind Visa TREASURE and FATA-Trans), which is cheaper and a stronger inductive bias than flattening every field into the token stream.

---

## Architecture

```
                       ┌─────────────────────── Anyscale ───────────────────────┐
 raw transactions ──►  │ [Ray Data]   static/dynamic tokenization (map_groups)   │
   (Parquet, S3)       │ [Ray Train]  masked-feature pretraining (PyTorch + DDP)│
                       │ [Ray Data]   batch embedding extraction (CPU read+GPU)  │
                       │ [XGBoost]    downstream fraud: raw vs FM vs fusion       │
                       │ [Ray Serve]  online embedding + fraud score (cached)     │
                       └─────────────────────────────────────────────────────────┘
```

Every stage is the **same code** from laptop to multi-node cluster — you change `ScalingConfig`, not your program.

---

**Walkthrough:** this notebook runs end-to-end at `mini` scale (a few thousand cards, a 2-layer model) so it completes in minutes on a small cluster. Flip `SCALE` to `small`/`full` and `USE_GPU=True` for the real distributed story.

## Get the code

```bash
git clone https://github.com/anyscale/templates && cd templates/templates/fintech_transaction_fm
```

## Step 1: Connect to the Ray Cluster

In an Anyscale Workspace, Ray is **pre-initialized** — no cluster setup, no Spark context. The install cell pulls the template's dependencies; on the GPU base image PyTorch is already present. Dependencies are light: ray, pytorch, xgboost and a few supporting libraries.

> In production you'd install from the generated `python_depset.lock`. Here we install from `requirements.txt` for portability.

In [ ]:
!pip install -q -r requirements.txt

In [1]:
import sys, os
DEMO_ROOT = os.path.abspath(os.getcwd())
if DEMO_ROOT not in sys.path:
    sys.path.insert(0, DEMO_ROOT)

import ray
ray.init(ignore_reinit_error=True, runtime_env={"working_dir": DEMO_ROOT})

from src.utils import print_cluster_resources
print_cluster_resources()

2026-06-24 21:12:15,981	INFO worker.py:1814 -- Connecting to existing Ray cluster at address: 10.0.188.86:6379...
2026-06-24 21:12:16,008	INFO worker.py:2003 -- Connected to Ray cluster. View the dashboard at https://session-nridlis6nyy2hi9qv4il5lcj6q.i.anyscaleuserdata.com 
2026-06-24 21:12:16,023	INFO packaging.py:691 -- Creating a file package for local module '/home/ray/default_cld_g54aiirwj1s8t9ktgzikqur41k/templates/templates/fintech_transaction_fm'.
2026-06-24 21:12:16,032	INFO packaging.py:463 -- Pushing file package 'gcs://_ray_pkg_aa1d3b983f06dd1d.zip' (0.17MiB) to Ray cluster...
2026-06-24 21:12:16,034	INFO packaging.py:476 -- Successfully pushed file package 'gcs://_ray_pkg_aa1d3b983f06dd1d.zip'.


Ray cluster resources:
  anyscale/cpu_only:true 1.0
  anyscale/node-group:head 1.0
  anyscale/provider:aws 1.0
  anyscale/region:us-west-2 1.0
  memory               34359738368.0
  object_store_memory  9603753984.0

Cluster nodes: 1
  10.0.188.86          alive=True  object_store_memory=9603753984.0, anyscale/node-group:head=1.0, anyscale/cpu_only:true=1.0, anyscale/region:us-west-2=1.0, anyscale/provider:aws=1.0, memory=34359738368.0


/home/ray/anaconda3/lib/python3.12/site-packages/ray/_private/worker.py:2051: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(


(autoscaler +3m15s) Tip: use `ray status` to view detailed cluster status. To disable these messages, set RAY_SCHEDULER_EVENTS=0.


## Step 2: Load & inspect transaction data

By default we use the **real IBM TabFormer dataset** (Padhi et al., ICASSP 2021 — the public benchmark for transaction foundation models: 24.4M transactions, ~6.1k cards, 0.12% fraud), downloaded once (~266MB) and sampled down to the scale's card budget. Each *card* has **static** fields (issuer, card type, BIN region, home state — derived where TabFormer lacks them) plus a time-ordered stream of transactions with **dynamic** fields (amount, merchant, MCC, hour, day-of-week) and a fraud label.

The loader also writes `splits.json` with **temporal 80/10/10 cutoffs** (train on the past, test on the most recent 10%) — the same evaluation protocol as NVIDIA's transaction-FM blueprint, so downstream numbers are comparable.

> Fully offline alternative: `python scripts/01_generate_data.py --source synthetic` generates schema-identical synthetic data.


In [3]:
import pandas as pd
from src.paths import SCALE_MAP, artifact_paths, get_demo_base_dir
from src.tabformer import prepare_tabformer

SCALE = "small"        # "small" / "full" for the distributed story
USE_GPU = True        # set True on a GPU cluster for train + embed

BASE_DIR = get_demo_base_dir()
paths = artifact_paths(BASE_DIR, SCALE)

if not (os.path.exists(paths["raw"]) and os.path.exists(paths["splits"])):
    prepare_tabformer(
        paths["raw"], paths["splits"],
        num_cards=SCALE_MAP[SCALE], source_dir=paths["source"],
    )

df = pd.read_parquet(paths["raw"])
print(f"{len(df):,} transactions / {df['card_id'].nunique():,} cards / fraud {df['is_fraud'].mean()*100:.3f}%")
df.head(4)

24,386,900 transactions / 6,139 cards / fraud 0.122%


,card_id,timestamp,amount,merchant_id,merchant_category,mcc,hour,day_of_week,is_fraud,issuer,bin_region,card_type,home_state
0,0,2002-09-01 06:21:00,134.09,3527213246127876953,retail,5300,6,6,0,UNKNOWN,UNKNOWN,swipe,CA
1,0,2002-09-01 06:42:00,38.48,-727612092139916043,grocery,5411,6,6,0,UNKNOWN,UNKNOWN,swipe,CA
2,0,2002-09-02 06:22:00,120.34,-727612092139916043,grocery,5411,6,0,0,UNKNOWN,UNKNOWN,swipe,CA
3,0,2002-09-02 17:45:00,128.95,3414527459579106770,retail,5651,17,0,0,UNKNOWN,UNKNOWN,swipe,CA


## Step 3: Tokenize with Ray Data — the static/dynamic split

This is the core idea. NVIDIA's blueprint flattens every transaction into ~12 tokens in one shared vocabulary, so a sequence of *N* transactions costs ~12*N* positions. We instead:

- embed **static** card-level fields **once** and broadcast them to every position (they never spend sequence length), and
- give each **dynamic** field its own embedding table, so each transaction is **one** position whose vector is the sum of its field embeddings.

The vocabulary is fully deterministic (fixed amount buckets + merchant hashing), so tokenization is a **stateless `map_groups`** with no global shuffle — exactly what Ray Data is built for. NVIDIA's RAPIDS path is single-GPU; this scales across the cluster.

Two representation choices live here. **Positions are time-aware**: alongside the ordinal position we embed the log-bucketed *gap since the previous transaction*, because for transactions *when* matters more than ordinal slot. And **amount uses bucketing** (`AMOUNT_MODE`), with an optional `"soft"` mode that blends the two adjacent bin embeddings so $86.99 and $87.01 don't land on unrelated vectors.

> **What you'll see while this runs** (and why it's fine):
> - **"Cluster does not have any available CPUs / job may hang"** — on a fresh or idle cluster the GPU worker is still launching; Ray Data waits and the warnings clear once it lands (~2 min). The head node intentionally schedules no work.
> - **Hash-shuffle aggregator warnings** — Ray's shuffle defaults assume a much bigger cluster; we right-size it per scale (`shuffle_partitions` + `max_hash_shuffle_aggregators` in `scripts/02_tokenize.py`).
> - **"Numba isn't available"** — `numba` (in `requirements.txt`) lets RayTurbo JIT the hash-partitioning kernel of this groupby; without it the shuffle falls back to slower Python.
> - **"Object store is configured to use only 28% of memory"** — set at cluster start on Anyscale; at this dataset size (~3GB) the default is plenty.


In [4]:
import json

import numpy as np
from ray.data.expressions import col
from src.tokenizer import SEQ_LEN_BY_SCALE, tokenize_dataset, write_vocab

seq_len = SEQ_LEN_BY_SCALE[SCALE]
with open(paths["splits"]) as f:
    splits = json.load(f)

ds = ray.data.read_parquet(paths["raw"])
tokenized = tokenize_dataset(
    ds, seq_len,
    train_end=splits["train_end"],   # temporal cutoffs -> pretrain never sees val/test
    val_end=splits["val_end"],
    normal_keep=0.005,               # downsample normal eval samples (all frauds kept)
    max_pretrain_windows=8,
    num_partitions=32,            # right-size the shuffle (Ray defaults to 200)
).materialize()

PRETRAIN_DROP = ["kind", "split", "label", "weight",
                 "raw_amount", "raw_hour", "raw_dow", "raw_mcc", "raw_ts"]
tokenized.filter(expr=col("kind") == "pretrain").drop_columns(PRETRAIN_DROP) \
    .write_parquet(paths["tokenized_pretrain"])
tokenized.filter(expr=col("kind") == "eval").drop_columns(["kind"]) \
    .write_parquet(paths["tokenized_eval"])
write_vocab(paths["vocab"])

tok = pd.read_parquet(paths["tokenized_eval"])
print(f"{len(tok):,} eval samples ({int((tok['label'] == 1).sum()):,} fraud), seq_len={seq_len}")
row = tok.iloc[0]
print("  static:", {c: int(row[c]) for c in tok.columns if c.startswith("s_")})
print("  amount-bucket tokens:", np.asarray(row["d_amount_bucket"])[-8:].tolist())
print("  attention mask:", np.asarray(row["attention_mask"])[-8:].tolist())

2026-06-24 21:21:56,810	WARNING plan.py:427 -- Warning: The Ray cluster currently does not have any available CPUs. The Dataset job will hang unless more CPUs are freed up. A common reason is that cluster resources are used by Actors or Tune trials; see the following link for more details: https://docs.ray.io/en/latest/data/data-internals.html#ray-data-and-tune
2026-06-24 21:21:56,812	INFO logging.py:416 -- Registered dataset logger for dataset dataset_3_0
2026-06-24 21:21:56,881	INFO streaming_executor.py:166 -- Starting execution of Dataset dataset_3_0. Full logs are in /tmp/ray/session_2026-06-24_20-51-04_749369_2727/logs/ray-data
2026-06-24 21:21:56,882	INFO streaming_executor.py:167 -- Execution plan of Dataset dataset_3_0: InputDataBuffer[Input] -> TaskPoolMapOperator[ListFiles] -> TaskPoolMapOperator[ReadFiles] -> HashShuffleOperator[Shuffle(key_columns=('card_id',), num_partitions=32)] -> TaskPoolMapOperator[MapBatches(tokenize_group)]
2026-06-24 21:21:56,885	WARNING hash_shuff

(autoscaler +9m42s) [autoscaler] [8CPU-32GB] Attempting to add 1 node to the cluster (increasing from 0 to 1).


2026-06-24 21:21:59,027	WARNING resource_manager.py:888 -- Cluster resources are not enough to run any task from TaskPoolMapOperator[ReadFiles]. The job may hang forever unless the cluster scales up.
2026-06-24 21:21:59,037	INFO logging_progress.py:174 -- ======= Running Dataset: dataset_3_0 =======
2026-06-24 21:21:59,038	INFO logging_progress.py:225 -- Total Progress: 0/?
2026-06-24 21:21:59,038	INFO logging_progress.py:227 -- Active & requested resources: 8/0 CPU, 0.0B/0.0B object store
2026-06-24 21:21:59,038	INFO logging_progress.py:181 -- 
2026-06-24 21:21:59,039	INFO logging_progress.py:231 -- ListFiles: 0/1
2026-06-24 21:21:59,039	INFO logging_progress.py:233 --   Tasks: 1; Actors: 0; Queued blocks: 0 (0.0B); Resources: 1.0 CPU, 0.0B object store
2026-06-24 21:21:59,040	INFO logging_progress.py:231 -- ReadFiles: 0/1
2026-06-24 21:21:59,041	INFO logging_progress.py:233 --   Tasks: 0 [backpressured:tasks(ResourceBudget)]; Actors: 0; Queued blocks: 0 (0.0B); Resources: 0.0 CPU, 0.

(autoscaler +9m47s) [autoscaler] [8CPU-32GB] Attempting to add 1 node to the cluster (increasing from 1 to 2).
(autoscaler +9m47s) [autoscaler] [8CPU-32GB|m5.2xlarge] [us-west-2c] [on-demand] Launched 1 instance.
(autoscaler +9m47s) [autoscaler] [8CPU-32GB|m5.2xlarge] [us-west-2c] [on-demand] Launched 1 instance.


2026-06-24 21:22:09,091	INFO logging_progress.py:174 -- ======= Running Dataset: dataset_3_0 =======
2026-06-24 21:22:09,091	INFO logging_progress.py:225 -- Total Progress: 0/?
2026-06-24 21:22:09,092	INFO logging_progress.py:227 -- Active & requested resources: 9/0 CPU, 0.0B/0.0B object store
2026-06-24 21:22:09,093	INFO logging_progress.py:181 -- 
2026-06-24 21:22:09,093	INFO logging_progress.py:231 -- ListFiles: 0/1
2026-06-24 21:22:09,093	INFO logging_progress.py:233 --   Tasks: 1; Actors: 0; Queued blocks: 0 (0.0B); Resources: 1.0 CPU, 0.0B object store
2026-06-24 21:22:09,094	INFO logging_progress.py:231 -- ReadFiles: 0/1
2026-06-24 21:22:09,094	INFO logging_progress.py:233 --   Tasks: 0 [backpressured:tasks(ResourceBudget)]; Actors: 0; Queued blocks: 0 (0.0B); Resources: 0.0 CPU, 0.0B object store
2026-06-24 21:22:09,095	INFO logging_progress.py:231 -- Shuffle(key_columns=('card_id',), num_partitions=32): 0/1
2026-06-24 21:22:09,095	INFO logging_progress.py:233 --   Tasks: 0 [ba

...

(autoscaler +14m32s) [autoscaler] Cluster upscaled to {96 CPU, 0 GPU}.


2026-06-24 21:26:52,528	INFO logging_progress.py:174 -- ======= Running Dataset: dataset_8_0 =======
2026-06-24 21:26:52,529	INFO logging_progress.py:225 -- Total Progress: 22/32
2026-06-24 21:26:52,529	INFO logging_progress.py:227 -- Active & requested resources: 10/80 CPU, 1.7KiB/44.9GiB object store
2026-06-24 21:26:52,530	INFO logging_progress.py:181 -- 
2026-06-24 21:26:52,530	INFO logging_progress.py:231 -- Filter(col('kind') == 'pretrain')->MapBatches(drop_columns)->Write: 22/32
2026-06-24 21:26:52,530	INFO logging_progress.py:233 --   Tasks: 10; Actors: 0; Queued blocks: 0 (0.0B); Resources: 10.0 CPU, 1.7KiB object store
2026-06-24 21:26:52,531	INFO logging_progress.py:192 -- ============================================


(autoscaler +14m42s) [autoscaler] Cluster upscaled to {160 CPU, 0 GPU}.


2026-06-24 21:27:02,574	INFO logging_progress.py:174 -- ======= Running Dataset: dataset_8_0 =======
2026-06-24 21:27:02,574	INFO logging_progress.py:225 -- Total Progress: 27/32
2026-06-24 21:27:02,575	INFO logging_progress.py:227 -- Active & requested resources: 5/160 CPU, 860.0B/89.7GiB object store
2026-06-24 21:27:02,575	INFO logging_progress.py:181 -- 
2026-06-24 21:27:02,576	INFO logging_progress.py:231 -- Filter(col('kind') == 'pretrain')->MapBatches(drop_columns)->Write: 27/32
2026-06-24 21:27:02,576	INFO logging_progress.py:233 --   Tasks: 5; Actors: 0; Queued blocks: 0 (0.0B); Resources: 5.0 CPU, 860.0B object store
2026-06-24 21:27:02,577	INFO logging_progress.py:192 -- ============================================
2026-06-24 21:27:03,325	INFO streaming_executor.py:294 -- ✔️  Dataset dataset_8_0 execution finished in 51.05 seconds
INFO:openlineage.client.client:OpenLineageClient will use `composite` transport
INFO:openlineage.client.transport.composite:Stopping OpenLineage C

1,064,721 eval samples (208,299 fraud), seq_len=128
  static: {'s_issuer': 15, 's_card_type': 7, 's_bin_region': 9, 's_home_state': 7}
  amount-bucket tokens: [8, 9, 7, 10, 9, 8, 8, 7]
  attention mask: [1, 1, 1, 1, 1, 1, 1, 1]


## Step 4: Pretrain with Ray Train (masked-feature modeling, DDP)

We pretrain by **masking dynamic-field tokens and predicting them** (the Stripe / Open-Banking objective — bidirectional context beats next-token for the fixed-window tasks fintech cares about). There's **one classification head per dynamic field**, and because those heads differ wildly in difficulty (merchant is ~2000-way, day-of-week is 9-way) we balance them with **uncertainty weighting** (Kendall & Gal) so the big head doesn't dominate.

The training loop is plain PyTorch; **Ray Train** handles worker setup, dataset sharding, **DDP** wrapping, checkpointing, and fault tolerance. The model fits one GPU at every scale, so we use DDP (data-parallel) and scale by adding workers — go from 1 CPU worker (here) to N GPU workers by changing only `num_workers` and `use_gpu`. (`use_fsdp` is available for when the model itself outgrows a GPU.)

In [5]:
from src.pretrain import pretrain

metrics = pretrain(
    tokenized_path=paths["tokenized_pretrain"],
    vocab_path=paths["vocab"],
    checkpoint_out=paths["checkpoint"],
    size=SCALE,
    max_len=seq_len,
    epochs=2,
    batch_size=64,
    num_workers=1,          # bump to 4-8 GPU workers at scale
    use_gpu=USE_GPU,
    use_fsdp=False,         # True for sharded multi-GPU training
    storage_base=BASE_DIR,  # shared storage — workers run on other nodes
)
print("final MLM loss:", round(metrics["mlm_loss"], 4))

(TrainController pid=17532) A run snapshot was found in storage folder at: '/mnt/cluster_storage/transaction-fm/ray_results/transaction_fm_pretrain'
(TrainController pid=17532) This snapshot contains a list of checkpoints reported via `ray.train.report` and will be loaded. This allows the latest checkpoint found in the snapshot to be accessible within your training function via `ray.train.get_checkpoint`.
(TrainController pid=17532) If you meant to start a brand new training job without any information about previous checkpoints found in this directory, please configure a new, unique `RunConfig(name)` or delete the existing folder at '/mnt/cluster_storage/transaction-fm/ray_results/transaction_fm_pretrain'.
(TrainController pid=17532) Requesting resources: {'GPU': 1} * 1
(TrainController pid=17532) [State Transition] INITIALIZING -> SCHEDULING.
(TrainController pid=17532) Attempting to start training worker group of size 1 with the following resources: [{'GPU': 1}] * 1


(autoscaler +16m32s) [autoscaler] [1xT4:8CPU-32GB] Attempting to add 1 node to the cluster (increasing from 0 to 1).
(autoscaler +16m32s) [autoscaler] [1xT4:8CPU-32GB|g4dn.2xlarge] [us-west-2c] [on-demand] Launched 1 instance.


(TrainController pid=17532) [FailurePolicy] RETRY
(TrainController pid=17532)   Source: controller
(TrainController pid=17532)   Error count: 1 (max allowed: inf)
(TrainController pid=17532) Error: Training failed due to controller error:
(TrainController pid=17532) The worker group startup timed out after 60.0 seconds waiting for 1 workers. Potential causes include: (1) temporary insufficient cluster resources while waiting for autoscaling (ignore this warning in this case), (2) infeasible resource request where the provided `ScalingConfig` cannot be satisfied), and (3) transient network issues. Set the RAY_TRAIN_WORKER_GROUP_START_TIMEOUT_S environment variable to increase the timeout.
(TrainController pid=17532) Traceback (most recent call last):
(TrainController pid=17532)   File "/home/ray/anaconda3/lib/python3.12/site-packages/ray/train/v2/_internal/execution/controller/controller.py", line 414, in _start_worker_group
(TrainController pid=17532)     self._worker_group = self.work

(autoscaler +17m32s) [autoscaler] Cluster upscaled to {168 CPU, 1 GPU}.


(RayTrainWorker pid=2939, ip=10.0.186.55) Setting up process group for: env:// [rank=0, world_size=1]
(TrainController pid=17532) Started training worker group of size 1: 
(TrainController pid=17532) - (ip=10.0.186.55, pid=2939) world_rank=0, local_rank=0, node_rank=0
(TrainController pid=17532) [State Transition] SCHEDULING -> RUNNING.
(RayTrainWorker pid=2939, ip=10.0.186.55) /tmp/ray/session_2026-06-24_20-51-04_749369_2727/runtime_resources/working_dir_files/_ray_pkg_aa1d3b983f06dd1d/src/model.py:79: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
(RayTrainWorker pid=2939, ip=10.0.186.55)   self.encoder = nn.TransformerEncoder(layer, num_layers=n_layers)
(RayTrainWorker pid=2939, ip=10.0.186.55) Moving model to device: cuda:0
(SplitCoordinator pid=18767) Registered dataset logger for dataset train_16_0
(SplitCoordinator pid=18767) Starting execution of Dataset train_16_0. Full logs are in /tmp/ray/session_2026-

...

(pid=18767) - split(1, equal=True):   0%|          | 0.00/1.00 [00:00<?, ? row/s]

(RayTrainWorker pid=2939, ip=10.0.186.55) Reporting training result 9: TrainingReport(checkpoint=None, metrics={'epoch': 0, 'mlm_loss': 11.174191390182612, 'acc_amount_bucket': 0.4375400713637537, 'ppl_amount_bucket': 4.314348101362399, 'acc_merchant_bucket': 0.1676492166978865, 'ppl_merchant_bucket': 114.81439352954446, 'acc_merchant_category': 0.3651920062036727, 'ppl_merchant_category': 5.34743625602731, 'acc_mcc': 0.2586790822642292, 'ppl_mcc': 13.636688620912663, 'acc_hour': 0.31310198987874216, 'ppl_hour': 10.74490526692792, 'acc_day_of_week': 0.14643651735214888, 'ppl_day_of_week': 7.087611352684376, 'acc_macro': 0.28143314729340557}, validation=False)


(autoscaler +23m52s) [autoscaler] Cluster resized to {168 CPU, 1 GPU}.
(autoscaler +23m57s) [autoscaler] Cluster upscaled to {264 CPU, 1 GPU}.
(autoscaler +23m57s) [autoscaler] Cluster upscaled to {360 CPU, 1 GPU}.


(SplitCoordinator pid=18767) ✔️  Dataset train_16_1 execution finished in 292.53 seconds
(SplitCoordinator pid=18767) INFO:openlineage.client.client:OpenLineageClient will use `composite` transport
(SplitCoordinator pid=18767) INFO:openlineage.client.transport.composite:Stopping OpenLineage CompositeTransport emission after the first successful delivery because `continue_on_success=False`. Transport that emitted the event: <HttpTransport(name=first, kind=http, priority=1)>
(RayTrainWorker pid=2939, ip=10.0.186.55) Checkpoint successfully created at: Checkpoint(filesystem=local, path=/mnt/cluster_storage/transaction-fm/ray_results/transaction_fm_pretrain/checkpoint_2026-06-24_21-41-05.492922)
(RayTrainWorker pid=2939, ip=10.0.186.55) Reporting training result 10: TrainingReport(checkpoint=Checkpoint(filesystem=local, path=/mnt/cluster_storage/transaction-fm/ray_results/transaction_fm_pretrain/checkpoint_2026-06-24_21-41-05.492922), metrics={'epoch': 1, 'mlm_loss': 7.868270545907175, 'ac

[pretrain] final mlm_loss=7.8683 macro_acc=0.293 -> /mnt/cluster_storage/transaction-fm/model/small/
final MLM loss: 7.8683


## Step 5: Batch embedding extraction with Ray Data

The recurring production job: score every customer to a fresh embedding. This is a heterogeneous **CPU-read + GPU-infer** workload that streams through one Ray Data pipeline — the model loads once per replica, batches stream through, output is written idempotently. This is the stage with no clean public reference, and where Ray clearly earns its keep.

In [6]:
from src.embed import extract_embeddings

extract_embeddings(
    tokenized_path=paths["tokenized_eval"],
    checkpoint_dir=paths["checkpoint"],
    output_path=paths["embeddings"],
    num_workers=1,          # scale out across GPU replicas at `full`
    use_gpu=USE_GPU,
)
emb = pd.read_parquet(paths["embeddings"])
dim = np.asarray(emb["embedding"].iloc[0]).shape[0]
print(f"{len(emb):,} transaction-window embeddings, dim={dim}")

2026-06-24 21:41:09,961	INFO logging.py:416 -- Registered dataset logger for dataset dataset_21_0
2026-06-24 21:41:09,966	INFO streaming_executor.py:166 -- Starting execution of Dataset dataset_21_0. Full logs are in /tmp/ray/session_2026-06-24_20-51-04_749369_2727/logs/ray-data
2026-06-24 21:41:09,967	INFO streaming_executor.py:167 -- Execution plan of Dataset dataset_21_0: InputDataBuffer[Input] -> TaskPoolMapOperator[ListFiles] -> TaskPoolMapOperator[ReadFiles] -> ActorPoolMapOperator[MapBatches(EmbeddingExtractor)] -> TaskPoolMapOperator[Write]
{"asctime":"2026-06-24 21:41:10,002","levelname":"E","message":"Actor with class name: 'MapWorker(MapBatches(EmbeddingExtractor))' and ID: 'd2d31589189f16cc3588515f03000000' has constructor arguments in the object store and max_restarts > 0. If the arguments in the object store go out of scope or are lost, the actor restart will fail. See https://github.com/ray-project/ray/issues/53727 for more details.","filename":"core_worker.cc","lineno":

(MapWorker(MapBatches(EmbeddingExtractor)) pid=5014, ip=10.0.186.55) [embed] loaded TransactionFM on cuda


(MapWorker(MapBatches(EmbeddingExtractor)) pid=5014, ip=10.0.186.55) /tmp/ray/session_2026-06-24_20-51-04_749369_2727/runtime_resources/working_dir_files/_ray_pkg_aa1d3b983f06dd1d/src/embed.py:56: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
(MapWorker(MapBatches(EmbeddingExtractor)) pid=5014, ip=10.0.186.55)   return torch.as_tensor(v, dtype=dtype, device=self.device)
2026-06-24 21:41:20,128	INFO logging_progress.py:174 -- ======= Running Dataset: dataset_21_0 =======
2026-06-24 21:41:20,129	INFO logging_progress.py:225 -- Total Progress: 1/200
2026-06-24 21:41:20,129	INFO logging_progress.py:227 -- Active & request

[embed] wrote customer embeddings -> /mnt/cluster_storage/transaction-fm/embeddings/small/


2026-06-24 21:47:41,530	INFO streaming_executor.py:294 -- ✔️  Dataset dataset_24_0 execution finished in 1.67 seconds
INFO:openlineage.client.client:OpenLineageClient will use `composite` transport
INFO:openlineage.client.transport.composite:Stopping OpenLineage CompositeTransport emission after the first successful delivery because `continue_on_success=False`. Transport that emitted the event: <HttpTransport(name=first, kind=http, priority=1)>


[embed] health: mean_pairwise_cos=0.102 (→1 = collapse), mean_feature_var=3.2168, n=2000
3,650,472 transaction-window embeddings, dim=256


(autoscaler +1h10m52s) [autoscaler] Cluster resized to {264 CPU, 1 GPU}.
(autoscaler +1h12m52s) [autoscaler] Downscaling node i-08b1b39d146da3bf0 (node IP: 10.0.129.38) due to node idle termination.
(autoscaler +1h12m52s) [autoscaler] Downscaling node i-0b098d820f75ae99a (node IP: 10.0.172.94) due to node idle termination.
(autoscaler +1h12m52s) [autoscaler] Downscaling node i-0d107b304ad0ed26f (node IP: 10.0.158.161) due to node idle termination.
(autoscaler +1h12m52s) [autoscaler] Downscaling node i-0d53b69ea331e129d (node IP: 10.0.182.3) due to node idle termination.
(autoscaler +1h12m52s) [autoscaler] Downscaling node i-0f201cf44677f9e23 (node IP: 10.0.191.252) due to node idle termination.
(autoscaler +1h12m52s) [autoscaler] Downscaling node i-0d977f8369b6dc55a (node IP: 10.0.150.244) due to node idle termination.
(autoscaler +1h12m52s) [autoscaler] Downscaling node i-030e488a62839c764 (node IP: 10.0.172.85) due to node idle termination.
(autoscaler +1h12m52s) [autoscaler] Cluster

: 

## Step 6: Downstream fraud — raw vs FM vs fusion

The headline result, evaluated with the **NVIDIA transaction-FM blueprint protocol**: temporal 80/10/10 split, per-transaction last-event fraud labels, AUC-ROC + PR-AUC at natural fraud prevalence (downsampled normals are importance-weighted back). Same XGBoost recipe, three feature sets:

1. **raw** — the target transaction's tabular fields (what you have today)
2. **fm** — the FM embedding of the history window only
3. **fusion** — embedding ++ raw features (Nubank's joint fusion)

The lift of (2) and (3) over (1) is the case for a transaction FM. *(At `mini` scale — 2 CPU epochs, a 2-layer model — expect fusion ≈ raw; the gap opens with the `small`/`full` GPU pretrain.)*


In [ ]:
from src.downstream import run_downstream, print_summary

summary = run_downstream(paths["embeddings"], paths["downstream"])
print_summary(summary)

## Step 7: Online serving with Ray Serve

The default production path is batch (Step 5) → feature store → XGBoost. But fraud also needs a real-time path, so we ship a **Ray Serve** deployment that mirrors the two-tier pattern real shops use: it **caches static (card-level) embeddings** and runs the transformer only over the recent dynamic window, returning an embedding + fraud score in one call. Autoscales 1→4 replicas.

In [ ]:
import requests
from ray import serve
from src.serve import build_app
from src.utils import sample_serve_payload

serve.run(build_app(paths["checkpoint"]), name="txn-fm")
payload = sample_serve_payload(paths["tokenized_eval"])
resp = requests.post("http://localhost:8000/", json=payload, timeout=30).json()
print("card_id   :", resp["card_id"])
print("embedding :", [round(x, 3) for x in resp["embedding"][:6]], "...")
print("fraud_score:", round(resp["fraud_score"], 4))
serve.shutdown()

## Step 8: Observability & fault tolerance

**Observability** (built into Anyscale)
- **Ray Dashboard** — watch data stream through tokenize/embed stages, per-worker throughput, GPU utilization.
- **Ray Train reports** — per-epoch MLM loss, checkpoint lineage.
- **Serve metrics** — latency, replica autoscaling, ongoing requests.

**Fault tolerance** (built into Ray)
- Ray Data retries failed batches per-partition — no full restart.
- Ray Train checkpoints let pretraining resume after a node loss.
- Streaming + backpressure keeps memory bounded across stages.

## Step 9: Path to production

The same code scales up by changing config, and runs as a scheduled **Anyscale Job**. `scripts/run_pipeline.py` wraps all six stages (data -> tokenize -> pretrain -> embed -> downstream -> validate) in one command:

```bash
# Full pipeline as a Job (GPU workers, autoscaling):
anyscale job submit --config-file job_config.yaml

# Larger scale:
anyscale job submit --config-file job_config.yaml -- python scripts/run_pipeline.py --scale full
```

| Stage | Ray primitive | Scale knob |
|-------|---------------|-----------|
| Tokenize | Ray Data `map_groups` | partitions / cluster size |
| Pretrain | Ray Train + DDP | `num_workers`, `use_gpu` |
| Batch embed | Ray Data `map_batches` | `num_workers`, `num_gpus` |
| Online serve | Ray Serve | replica autoscaling |


## Validate

A quick end-to-end check that every stage produced sane artifacts.

In [ ]:
from scripts.validate_results import validate_pipeline, print_report
print_report(validate_pipeline(paths))